# Demonstration of MUSE

This is a demonstration of MUSE analysis on a multi-modality simulated data.

Feng Bao @ Altschuler & Wu Lab @ UCSF 2022.

Software provided as is under MIT License.

## Import packages

In [1]:
import multi_muse_sc as muse
import simulation_tool.multi_modal_simulation as simulation

import phenograph
from sklearn.decomposition import PCA
import numpy as np
from sklearn.metrics.cluster import adjusted_rand_score
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
np.random.seed(0)

## Generate simulation data


Simulation parameters

In [2]:
latent_dim = 100
num_cluster = 10
sample_size = 1000
latent_code_dim = 30
observed_data_dim = 500
sigma_1 = 0.1  
sigma_2 = 0.1
decay_coef_1 = 0.5 
decay_coef_2 = 0.1
merge_prob = 0.7



Use simulation tool to generate multi-modality data

In [3]:
data = simulation.multi_modal_simulator(
    num_cluster,
    sample_size,
    observed_data_dim,
    observed_data_dim,
    latent_code_dim,
    sigma_1,
    sigma_2,
    decay_coef_1,
    decay_coef_2,
    merge_prob,
)
data_a = data["data_a_dropout"]
data_b = data["data_b_dropout"]
label_a = data["data_a_label"]
label_b = data["data_b_label"]
label_true = data["true_cluster"]

## Analyses based on single modality

Learn features from single modality

In [ ]:
data_a

In [5]:
data_ab = (data_a + 1) * (data_b + 1)
# data_ab = data_a * data_b

view_a_feature = PCA(n_components=latent_dim, random_state=42).fit_transform(data_a)
view_b_feature = PCA(n_components=latent_dim, random_state=42).fit_transform(data_b)
view_ab_feature = PCA(n_components=latent_dim, random_state=42).fit_transform(data_ab)

In [ ]:
print(f"data_a shape: {data_a.shape}")
print(f"data_b shape: {data_b.shape}")
print(f"view_a_feature shape: {view_a_feature.shape}")
print(f"view_b_feature shape: {view_b_feature.shape}")

Perform clustering using PhenoGraph

In [ ]:
view_a_label, _, _ = phenograph.cluster(view_a_feature, random_state=42)
view_b_label, _, _ = phenograph.cluster(view_b_feature, random_state=42)
view_ab_label, _, _ = phenograph.cluster(view_ab_feature, random_state=42)

## Combined analysis using MUSE

MUSE learns the joint latent representation

In [8]:
X_embedded = TSNE(n_components=2, random_state=42).fit_transform(view_a_feature)
Y_embedded = TSNE(n_components=2, random_state=42).fit_transform(view_b_feature)
XY_embedded = TSNE(n_components=2, random_state=42).fit_transform(view_ab_feature)

In [ ]:
plt.rcParams['font.size'] = 12
plt.figure(figsize=(12, 12))
plt.subplot(2, 2, 1)
# 遍历每一个类别
for i in np.unique(view_ab_label):
    # 找到所有类别为i的索引
    idx = np.nonzero(view_ab_label == i)[0]
    # 将类别为i的点画成散点图
    plt.scatter(X_embedded[idx, 0], X_embedded[idx, 1])
    # 设置标题
    plt.title('x (x + y org), ARI = %01.3f' % adjusted_rand_score(view_ab_label, view_a_label))

plt.subplot(2, 2, 2)
for i in np.unique(view_ab_label):
    idx = np.nonzero(view_ab_label == i)[0]
    plt.scatter(Y_embedded[idx, 0], Y_embedded[idx, 1])
    plt.title('y (x + y org), ARI = %01.3f' % adjusted_rand_score(view_ab_label, view_b_label))

plt.subplot(2, 2, 3)
for i in np.unique(view_ab_label):
    idx = np.nonzero(view_ab_label == i)[0]
    plt.scatter(XY_embedded[idx, 0], XY_embedded[idx, 1])
    plt.title('x + y (x + y org), ARI = %01.3f' % adjusted_rand_score(view_ab_label, view_ab_label))

plt.subplot(2, 2, 4)
for i in np.unique(label_true):
    idx = np.nonzero(label_true == i)[0]
    plt.scatter(XY_embedded[idx, 0], XY_embedded[idx, 1])
    plt.title('x + y (true), ARI = %01.3f' % adjusted_rand_score(view_ab_label, label_true))
plt.show()

In [ ]:
outputs = muse.dual_muse_fit_predict(
    [data_a, data_b],
    [data_ab],
    [view_a_label, view_b_label],
    [view_ab_label],
    latent_dim=100,
    n_epochs=500,
    info_nce_lambda=10,
    weight_penalty=1,
    triplet_lambda=1,
)

## Perform clustering
PhenoGraph clustering

In [ ]:
latent_sc, latent_st = outputs[0], outputs[1]
muse_label_sc, _, _ = phenograph.cluster(latent_sc, random_state=42)
muse_label_st, _, _ = phenograph.cluster(latent_st, random_state=42)
# 用kmeans聚类
from sklearn.cluster import KMeans
n_clusters = np.unique(view_ab_label).shape[0]
print(f"n_clusters: {n_clusters}")
kmeans_sc = KMeans(n_clusters=n_clusters).fit_predict(latent_sc)
kmeans_st = KMeans(n_clusters=n_clusters).fit_predict(latent_st)


In [ ]:
reconstruct_sc = (outputs[2])
reconstruct_st = (outputs[3])
encoded_sc = (outputs[4])
encoded_st = (outputs[5])
print(f"latent_sc: {latent_sc.shape}")
print(f"latent_st: {latent_st.shape}")
print(f"reconstruct_sc: {reconstruct_sc.shape}")
print(f"reconstruct_st: {reconstruct_st.shape}")
print(f"encoded_sc: {encoded_sc.shape}")
print(f"encoded_st: {encoded_st.shape}")

## Visualization of latent spaces 
Latent spaces of single-modality features or MUSE features were visualized using tSNE, with ground truth cluster labels.

Cluster accuries were quantified using adjusted Rand index (ARI). ARI = 1 indicates perfectly discover true cell identities.

In [ ]:
import umap
# 降维
kmeans_st = KMeans(n_clusters=6, random_state=42).fit_predict(latent_sc)
Y_tsne = TSNE(n_components=2, random_state=42).fit_transform(latent_sc)
Y_umap = umap.UMAP(n_components=2, random_state=42).fit_transform(latent_sc)

In [ ]:

print(np.unique(view_ab_label))
print(np.unique(muse_label_sc))
print(np.unique(muse_label_st))


# 创建图形
fig = plt.figure(figsize=(12, 6))

# TSNE 2D子图
ax1 = fig.add_subplot(121)
for i in np.unique(kmeans_st):
    idx = np.nonzero(kmeans_st == i)[0]
    ax1.scatter(Y_tsne[idx, 0], Y_tsne[idx, 1], label=f'Cluster {i}')
ax1.set_title('t-SNE 2D (ARI = %.3f)' % adjusted_rand_score(view_ab_label, kmeans_st))
ax1.set_xlabel('t-SNE1')
ax1.set_ylabel('t-SNE2')
ax1.legend()

# UMAP 2D子图
ax2 = fig.add_subplot(122)
for i in np.unique(kmeans_st):
    idx = np.nonzero(kmeans_st == i)[0]
    ax2.scatter(Y_umap[idx, 0], Y_umap[idx, 1], label=f'Cluster {i}')
ax2.set_title('UMAP 2D (ARI = %.3f)' % adjusted_rand_score(view_ab_label, kmeans_st))
ax2.set_xlabel('UMAP1')
ax2.set_ylabel('UMAP2')
ax2.legend()

# 调整布局
plt.tight_layout()
plt.show()

In [15]:
# # 3D可视化
# import plotly.express as px
# import plotly.graph_objects as go
# from plotly.subplots import make_subplots

# # 降维
# Y_tsne = TSNE(n_components=3, random_state=42).fit_transform(latent_st)
# Y_umap = umap.UMAP(n_components=3, random_state=42).fit_transform(latent_st)
# kmeans_st_tsne = KMeans(n_clusters=6, random_state=42).fit_predict(Y_tsne)
# kmeans_st_umap = KMeans(n_clusters=6, random_state=42).fit_predict(Y_umap)

# # 创建子图
# fig = make_subplots(
#     rows=1, cols=2,
#     specs=[[{'type': 'scene'}, {'type': 'scene'}]],
#     subplot_titles=('t-SNE 3D', 'UMAP 3D')
# )

# # 添加t-SNE散点图
# for i in np.unique(kmeans_st_tsne):
#     idx = np.nonzero(kmeans_st_tsne == i)[0]
#     fig.add_trace(
#         go.Scatter3d(
#             x=Y_tsne[idx, 0],
#             y=Y_tsne[idx, 1],
#             z=Y_tsne[idx, 2],
#             mode='markers',
#             marker=dict(size=4),
#             name=f'Cluster {i}',
#         ),
#         row=1, col=1
#     )

# # 添加UMAP散点图
# for i in np.unique(kmeans_st_umap):
#     idx = np.nonzero(kmeans_st_umap == i)[0]
#     fig.add_trace(
#         go.Scatter3d(
#             x=Y_umap[idx, 0],
#             y=Y_umap[idx, 1],
#             z=Y_umap[idx, 2],
#             mode='markers',
#             marker=dict(size=4),
#             name=f'Cluster {i}',
#             showlegend=False  # 不重复显示图例
#         ),
#         row=1, col=2
#     )

# # 更新布局
# fig.update_layout(
#     title_text=f"3D Visualization (ARI = {adjusted_rand_score(view_ab_label, kmeans_st):.3f})",
#     width=1500,
#     height=600,
# )

# # 更新每个子图的坐标轴标签
# fig.update_scenes(
#     row=1, col=1,
#     xaxis_title="t-SNE1",
#     yaxis_title="t-SNE2",
#     zaxis_title="t-SNE3"
# )

# fig.update_scenes(
#     row=1, col=2,
#     xaxis_title="UMAP1",
#     yaxis_title="UMAP2",
#     zaxis_title="UMAP3"
# )

# # 显示图形
# fig.show()

# # 可选：保存为HTML文件
# # fig.write_html("3d_visualization.html")

In [16]:
X_embedded = TSNE(n_components=2, random_state=42).fit_transform(latent_sc)
Y_embedded = TSNE(n_components=2, random_state=42).fit_transform(latent_st)

In [ ]:
plt.figure(figsize=(12, 11))
plt.subplot(2, 2, 1)
for i in np.unique(view_ab_label):
    idx = np.nonzero(view_ab_label == i)[0]
    plt.scatter(X_embedded[idx, 0], X_embedded[idx, 1])
    plt.title('latent SC (x + y org), ARI = %01.3f' % adjusted_rand_score(view_ab_label, muse_label_sc))

plt.subplot(2, 2, 2)
for i in np.unique(view_ab_label):
    idx = np.nonzero(view_ab_label == i)[0]
    plt.scatter(Y_embedded[idx, 0], Y_embedded[idx, 1])
    plt.title('latent ST (x + y org), ARI = %01.3f' % adjusted_rand_score(view_ab_label, muse_label_st))

plt.subplot(2, 2, 3)
for i in np.unique(label_true):
    idx = np.nonzero(label_true == i)[0]
    plt.scatter(X_embedded[idx, 0], X_embedded[idx, 1])
    plt.title('latent SC (true), ARI = %01.3f' % adjusted_rand_score(label_true, muse_label_sc))

plt.subplot(2, 2, 4)
for i in np.unique(label_true):
    idx = np.nonzero(label_true == i)[0]
    plt.scatter(Y_embedded[idx, 0], Y_embedded[idx, 1])
    plt.title('latent ST (true), ARI = %01.3f' % adjusted_rand_score(label_true, muse_label_st))

In [ ]:
print('phenograph SC and ST ARI: ', adjusted_rand_score(muse_label_sc, muse_label_st))
print('kmeans SC and ST ARI: ', adjusted_rand_score(kmeans_sc, kmeans_st))

In [19]:
# import os
# os.makedirs('latent_sc_st', exist_ok=True)
# np.save(f'latent_sc_st/latent_sc.npy', latent_sc)
# np.save(f'latent_sc_st/latent_st.npy', latent_st)

In [ ]:
import sim_tangram.train_tangram as tg_cca
import torch

# tangram
tangram_sim, _ = tg_cca.read_csv_and_run_tangram(
    [torch.tensor(latent_sc, device='cuda:0')],
    [torch.tensor(latent_st, device='cuda:0')],
    num_epochs=500,
    print_interval=500,
    lambda_g1=1,
    lambda_g2=1,
)

In [ ]:
# 用cosine_similarity驱动的tangram进行计算
import cos_tg.train_tangram as tg_cos

# tangram
cos_sim_tg, _ = tg_cos.read_csv_and_run_tangram(
    [torch.tensor(latent_sc, device='cuda:0')],
    [torch.tensor(latent_st, device='cuda:0')],
    num_epochs=500,
    print_interval=500,
    lambda_g1=1,
    lambda_g2=0,
)

In [ ]:
# 计算cosine_similarity
from sklearn.metrics.pairwise import cosine_similarity
cos = cosine_similarity(latent_sc, latent_st)
cos_mapping = cos.argmax(axis=1)
for i in range(len(cos_mapping)):
    if i < 30:
        if i % 5 == 0:
            print()
        print(f"{i:2d}: {cos_mapping[i]:4d}", end="\t")
print()
print('*'*50)

# 检验cosine_similarity驱动的tangram
cos_tg_mapping = cos_sim_tg.argmax(axis=1)
for i in range(len(cos_tg_mapping)):
    if i < 30:
        if i % 5 == 0:
            print()
        print(f"{i:2d}: {cos_tg_mapping[i]:4d}", end="\t")
print()
print('*'*50)

# 检验CCA驱动的tangram
tangram_mapping = tangram_sim.argmax(axis=1)
for i in range(len(tangram_mapping)):
    if i < 30:
        if i % 5 == 0:
            print()
        print(f"{i:2d}: {tangram_mapping[i]:4d}", end="\t")
print()
print('*'*50)


In [ ]:
# 输出cos对角线的元素
for i in range(len(cos_mapping)):
    if i < 30:
        print(cos[i, i], end="\t")

In [ ]:
cos

In [ ]:
# 计算acc
gt = np.array(range(sample_size))
cos_acc = np.sum(cos_mapping == gt) / sample_size
tangram_acc = np.sum(tangram_mapping == gt) / sample_size
cos_tg_acc = np.sum(cos_tg_mapping == gt) / sample_size
print("cos accuracy:", cos_acc)
print("tangram accuracy:", tangram_acc)
print("cos_tg accuracy:", cos_tg_acc)
